In [ ]:
import sys
sys.path.append('/data/users/quentin/package_paper/')
import scClone2DR as sccdr
import matplotlib.pyplot as plt
import numpy as np
from copy import deepcopy
import pickle
import torch
from tqdm import tqdm
import os
import pandas as pd
import plotly.io as pio
np.float_ = np.float64
rootpath = '/data/users/quentin/package_paper/experiments/paper_results'

COHORT = 'melanoma'
gene_set_collections = ['geneOncoKB','gene','c6','hallmarks', 'c2_pid']#, 'c2_kegg_medicus']
cohort2clonemodes = {'melanoma': ["scatrex",'scatrex_rawcounts_scvi', 'phenograph'], 'aml':['phenograph']}
gene_set_collection = gene_set_collections[3]
clonemode = cohort2clonemodes[COHORT][0]
mode_features = 'metacells_{0}_{1}'.format(gene_set_collection, clonemode)

directory = os.path.join(rootpath,COHORT)
if not os.path.exists(directory):
    os.makedirs(directory)
pathsave = os.path.join(rootpath,COHORT,gene_set_collection+'_'+clonemode)


if COHORT=='melanoma':
    path_rna = '/data/users/04_share_reanalysis_results/melanoma_2025/02_atypical_removed_preprocessing/{0}/'.format(mode_features)
    path_fastdrug = '/data/users/04_share_reanalysis_results/melanoma_2025/MEL_CNN_abundances_no_plate_effect_correction.csv'
    concentration_DMSO = '100'
    concentration_drug = '5'
    path_info_cohort = None
elif COHORT=='aml':
    concentration_DMSO = '200'
    concentration_drug = '10'
    path_rna = '/data/users/04_share_reanalysis_results/aml_2025/02_atypical_removed_preprocessing/{0}/'.format(mode_features)
    path_fastdrug = '/data/users/04_share_reanalysis_results/01_aml/AML_PCY_cell_numbers_no_plate_effect_correction.csv'
    path_info_cohort = '/data/users/04_share_reanalysis_results/01_aml/2024-08-15_aml_overview_scRNA.tsv'

model = sccdr.models.scClone2DR(path_fastdrug=path_fastdrug, path_rna=path_rna, path_info_cohort=path_info_cohort, type_guide="lowrank_MVN", rank=10)
data_ref = model.get_real_data(concentration_DMSO=concentration_DMSO, concentration_drug=concentration_drug)
selected_drugs = model.FD.selected_drugs

In [ ]:
with open(os.path.join(rootpath,'{0}/{1}_{2}/params_svi.pkl'.format(COHORT, gene_set_collection, clonemode)), 'rb') as handle:
    params_svi = pickle.load(handle)

with open(os.path.join(rootpath,'{0}/{1}_{2}/posterior_mean_params.pkl'.format(COHORT, gene_set_collection, clonemode)), 'rb') as handle:
    res = pickle.load(handle)

In [ ]:
df = pd.read_csv('/data/users/quentin/package_paper/experiments/survival_analysis/ord_longi.csv')
df[df['tupro_id']=="MANOFYB"][['days', 'prev_state', 'state']]

In [ ]:
30*3*3

In [ ]:
drugs = model.FD.selected_drugs

In [ ]:
idxs_drugs_cli = [i for i,drug in enumerate(drugs) if drug in alldrugs_clinical]

nb_drugs = []
for sampleID in range(len(model.sample_names)):
    el = np.sum(data_ref['n_r'][0,idxs_drugs_cli,sampleID].numpy()>10)
    nb_drugs.append(el)
np.mean(nb_drugs)

In [ ]:
# the fraction of patients who were given a drug that was tested
nb_drugs = []
selec_drugs = model.FD.selected_drugs
for sampleID, sample in enumerate((model.sample_names)):
    drugs_clini = sample2drugs_clini[sample]
    el = 0
    if drugs_clini != []:
        idxs_drugs_cli = pd.DataFrame(np.arange(len(selec_drugs)), index=selec_drugs).loc[drugs_clini]
        el = np.sum(data_ref['n_r'][0,idxs_drugs_cli.values.flatten(),sampleID].numpy()>10)>1
    nb_drugs.append(el)
np.mean(nb_drugs)

In [ ]:
# the fraction of patients who were given a drug that was tested
nb_drugs = []
selec_drugs = model.FD.selected_drugs
for sampleID, sample in enumerate((model.sample_names)):
    drugs_clini = sample2drugs_clini[sample]
    el = 1
    if drugs_clini != []:
        el = 0
    nb_drugs.append(el)
np.mean(nb_drugs)

In [ ]:
pd.DataFrame(np.arange(len(selec_drugs)), index=selec_drugs)

In [ ]:
idxs_drugs_cli = [i for i,drug in enumerate(drugs)]

nb_drugs = []
for sampleID in range(len(model.sample_names)):
    el = np.sum(data_ref['n_r'][0,idxs_drugs_cli,sampleID].numpy()>10)
    nb_drugs.append(el)
np.mean(nb_drugs)

In [ ]:
len(model.sample_names)

In [ ]:
len(sample2time2drugs.keys())

In [ ]:
## Loading the clinical data and the drug taken

import pickle
with open(r"/data/users/quentin/final_package/experiments/survival_analysis/sample2time2drugs.pkl", "rb") as input_file:
    sample2time2drugs = pickle.load(input_file)
    
    
    
samples = np.sort(list(sample2time2drugs.keys()))
sample2drugs_clini = {}
actual_dict = {}
alldrugs_clinical = []
for sample in samples:
    ls = sample + " & "
    drugs = []
    for time, tdrugs in sample2time2drugs[sample].items():
        drugs += tdrugs
        alldrugs_clinical += tdrugs
    drugs = list(set(drugs))
    sample2drugs_clini[sample] = drugs
    alldrugs_clinical = list(set(alldrugs_clinical))
    actual_dict[sample] = drugs
    if len(drugs)==0:
        ls += " &  0"
    else:
        ls += f"{','.join(drugs)} & {len(drugs)}"
    ls += "\\"
    print(ls)
        
print(alldrugs_clinical, len(alldrugs_clinical))

In [ ]:
cat2clusters = model.cat2clusters
pi = res['pi'].detach().numpy()
props = params_svi['proportions']
sample_names = model.sample_names

In [ ]:

ratio_pi = deepcopy(pi)
for idxdrug in range(pi.shape[0]):
    for sampleid in range(pi.shape[2]):
        learned_props = props[sampleid, :]
        healthy_survival = np.sum(
            np.nan_to_num(pi[idxdrug, :, sampleid] * learned_props)[cat2clusters['healthy']]
        )
        healthy_survival /= np.sum(learned_props[cat2clusters['healthy']])
        ratio_pi[idxdrug, :, sampleid] /= healthy_survival

scores = np.sum(
    np.nan_to_num(ratio_pi)[:, cat2clusters['tumor'], :] * 
    (props[:, cat2clusters['tumor']].T)[None, :, :], axis=1
)
scores /= np.tile((np.sum(props[:, cat2clusters['tumor']], axis=1)).reshape(1, -1), (ratio_pi.shape[0], 1))
scores = np.clip(scores, a_min=0, a_max=10)

for d in range(pi.shape[0]):
    pi[d, :, :][~(data_ref['masks']['RNA'])] = float('nan')
    ratio_pi[d, :, :][~(data_ref['masks']['RNA'])] = float('nan')


def pick_best_drug(mode_drug, mode_selection, rpi, ndrugs = 1):
    if mode_drug == "clinical_only":
        idxs = []
        drugs = deepcopy(alldrugs_clinical)
        for drug in drugs:
            idx_temp = [i for i, d in enumerate(selected_drugs) if d == drug]
            if len(idx_temp)==0:
                raise Exception(f"error: drug {drug} not found")
            else:
                idxs.append(idx_temp[0])
        rpi = rpi[idxs,:,:]
    else:
        drugs = deepcopy(selected_drugs)
    if mode_selection == "total_effect":
        stat = np.sum(np.nan_to_num(rpi[:,cat2clusters['tumor'],:] * (props.T)[None,cat2clusters['tumor'],:]), axis=1)
    else:
        stat = np.max(np.nan_to_num(rpi[:,cat2clusters['tumor'],:]), axis=1)
    if ndrugs==1:
        best_drugs_idxs = np.argmin(stat, axis=0)
        sample2best_drug = {sample_names[i]:[drugs[idx_drug]] for i, idx_drug in enumerate(best_drugs_idxs)}
    else:
        sample2best_drug = {}
        for i, sample in enumerate(sample_names):
            sample2best_drug[sample] = [drugs[idx_drug] for idx_drug in np.argsort(stat[:,i])[:ndrugs]]

    return sample2best_drug



def average_rank(mode_drug, mode_selection, rpi):
    if mode_drug == "clinical_only":
        idxs = []
        drugs = deepcopy(alldrugs_clinical)
        for drug in drugs:
            idx_temp = [i for i, d in enumerate(selected_drugs) if d == drug]
            if len(idx_temp)==0:
                raise Exception(f"error: drug {drug} not found")
            else:
                idxs.append(idx_temp[0])
        rpi = rpi[idxs,:,:]
    else:
        drugs = deepcopy(selected_drugs)
    if mode_selection == "total_effect":
        stat = np.sum(np.nan_to_num(rpi[:,cat2clusters['tumor'],:] * (props.T)[None,cat2clusters['tumor'],:]), axis=1)
    else:
        stat = np.max(np.nan_to_num(rpi), axis=1)
    sample2drug2rank = {}
    for i, sample in enumerate(sample_names):
        idxs_sorted_drugs = np.argsort(stat[:,i])
        sample2drug2rank[sample] = {} 
        for drug in actual_dict[sample]:
            idxs = [j for j in range(len(drugs)) if drugs[idxs_sorted_drugs[j]]==drug]
            assert len(idxs)==1
            sample2drug2rank[sample][drug] = idxs[0] + 1
    return sample2drug2rank


In [ ]:
    
def compare_pred_clinical(pred_dict, actual_dict, figure_name=None):

    import pandas as pd
    import seaborn as sns
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap
    import matplotlib.patches as mpatches

    # collect all patients
    patients = sorted(set(pred_dict) | set(actual_dict))

    # collect all drugs
    drugs = sorted(
        set(d for v in pred_dict.values() for d in v) |
        set(d for v in actual_dict.values() for d in v)
    )

    # create matrix
    df = pd.DataFrame(0, index=patients, columns=drugs)

    for p in patients:
        pred = set(pred_dict.get(p, []))
        actual = set(actual_dict.get(p, []))

        for d in pred:
            df.loc[p, d] += 1
        for d in actual:
            df.loc[p, d] += 2

    # color map
    cmap = ListedColormap([
        "#f0f0f0",  # none
        "#4C72B0",  # predicted only
        "#DD8452",  # given only
        "#55A868"   # both
    ])

    plt.figure(figsize=(14, 10))

    sns.heatmap(
        df,
        cmap=cmap,
        vmin=0,
        vmax=3,          # <-- FIX
        linewidths=0.5,
        linecolor="lightgray",
        cbar=False
    )

    plt.xlabel("Drug", fontsize=13)
    plt.ylabel("Sample", fontsize=13)

    plt.xticks(rotation=90, ha="center", fontsize=13)
    plt.yticks(fontsize=11)

    legend_handles = [
        mpatches.Patch(color="#f0f0f0", label="None"),
        mpatches.Patch(color="#4C72B0", label="Highest ΔLOR only"),
        mpatches.Patch(color="#DD8452", label="Administered only"),
        mpatches.Patch(color="#55A868", label="Both highest ΔLOR and administered")
    ]

    plt.legend(
        handles=legend_handles,
        bbox_to_anchor=(.5, 1.1),
        title="Treatment status",
        title_fontsize=15,
        loc="upper center",
        borderaxespad=0,
        fontsize=15,
        ncol=4
    )

    plt.tight_layout()

    if figure_name is not None:
        plt.savefig(f"{figure_name}.png", dpi=300, bbox_inches="tight")

    plt.show()
        
for mode_drug in ['all', 'clinical_only']:
    for mode_selection in ['total_effect', 'most_resistant']:
        print(mode_drug, mode_selection)
        sample2best_drug = pick_best_drug(mode_drug, mode_selection, ratio_pi)
        compare_pred_clinical(sample2best_drug, actual_dict)

In [ ]:
def compare_pred_clinical(pred_dict, actual_dict, sample2drug2rank, figure_name=None):

    import pandas as pd
    import seaborn as sns
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap
    import matplotlib.patches as mpatches

    # collect all patients
    patients = sorted(set(pred_dict) | set(actual_dict))

    # collect all drugs
    drugs = sorted(
        set(d for v in pred_dict.values() for d in v) |
        set(d for v in actual_dict.values() for d in v)
    )

    # matrices
    df = pd.DataFrame(0, index=patients, columns=drugs)
    annot_df = pd.DataFrame("", index=patients, columns=drugs)

    for p in patients:
        pred = set(pred_dict.get(p, []))
        actual = set(actual_dict.get(p, []))

        for d in pred:
            df.loc[p, d] += 1

        for d in actual:
            df.loc[p, d] += 2

            # add rank annotation
            if p in sample2drug2rank and d in sample2drug2rank[p]:
                annot_df.loc[p, d] = str(sample2drug2rank[p][d])
                if df.loc[p, d] == 3: # in case two drugs are on par (o avoid confusion and not having 1 potentially)
                    annot_df.loc[p, d] = 1

    # color map
    cmap = ListedColormap([
        "#f0f0f0",  # none
        "#4C72B0",  # predicted only
        "#DD8452",  # administered only
        "#55A868"   # both
    ])

    plt.figure(figsize=(14, 12))

    sns.heatmap(
        df,
        cmap=cmap,
        vmin=0,
        vmax=3,
        linewidths=0.5,
        linecolor="lightgray",
        cbar=False,
        annot=annot_df,
        fmt="",
        annot_kws={"fontsize":13, "color":"black"}
    )

    plt.xlabel("Drug", fontsize=16)
    plt.ylabel("Sample", fontsize=16)

    plt.xticks(rotation=90, ha="center", fontsize=16)
    plt.yticks(fontsize=12)

    legend_handles = [
        mpatches.Patch(color="#4C72B0", label="Recommended by scClone2DR only"),
        mpatches.Patch(color="#DD8452", label="Clinically administered only"),
        mpatches.Patch(color="#55A868", label="Recommended by scClone2DR and clinically administered")
    ]

    plt.legend(
        handles=legend_handles,
        bbox_to_anchor=(.5, 1.18),
        title="Treatment status",
        title_fontsize=15,
        loc="upper center",
        borderaxespad=0,
        fontsize=16,
        ncol=1
    )

    plt.tight_layout()

    if figure_name is not None:
        plt.savefig(f"{figure_name}.png", dpi=300, bbox_inches="tight")

    plt.show()
    
for mode_drug in ['all', 'clinical_only']:
    for mode_selection in ['total_effect', 'most_resistant']:
        print(mode_drug, mode_selection)
        sample2best_drug = pick_best_drug(mode_drug, mode_selection, ratio_pi)

        sample2drug2rank = average_rank(mode_drug, mode_selection, ratio_pi)
        compare_pred_clinical(sample2best_drug, actual_dict, sample2drug2rank, figure_name=f"{mode_drug}_{mode_selection}")


In [ ]:
import os
path = '/data/users/quentin/package_paper/experiments/survival_analysis/'
df = pd.read_csv(os.path.join(path, 'ord_longi.csv'), index_col=0)
sample2last_state = {}
for sample in sample_names:
    subset = df[df['tupro_id'] == sample]

    if not subset.empty:
        sample2last_state[sample] = subset['state'].iloc[-1]
    else:
        sample2last_state[sample] = None

In [ ]:
sample2last_state

In [ ]:
ls_states = []
ls_ranks = []
sample2drug2rank = average_rank("clinical_only", "total_effect", ratio_pi)

for sample in sample_names:
    if sample in sample2last_state.keys():
        try:
            ls_ranks.append(np.min(list(sample2drug2rank[sample].values())))
            ls_states.append(sample2last_state[sample])
        except:
            pass

In [ ]:
plt.scatter(ls_ranks,ls_states)